# Final Figure Reproduction Instructions

This notebook reproduces the seven figures used in `REPORT.md` from small bundled fixtures. It does not require private training data, source documents, model checkpoints, or API keys.

Before running, install the ML/figure dependencies from the repository root:

```bash
python -m pip install -e ".[ml]"
```

## Figures

| Figure | Output file | Data source |
|--------|-------------|-------------|
| 1 | `figure1_training_dynamics.png` | `data/fixtures/trainer_state.json` |
| 2 | `figure2_prompt_length_distribution.png` | `data/fixtures/final_report_metrics.json` |
| 3 | `figure3_checkpoint_comparison.png` | `data/fixtures/final_report_metrics.json` |
| 4 | `figure4_qa_triage_3systems.png` | `data/fixtures/final_report_metrics.json` |
| 5 | `figure5_source_attribution.png` | `data/fixtures/final_report_metrics.json` |
| 6 | `figure6_cost_quality.png` | `data/fixtures/final_report_metrics.json` |
| 7 | `figure7_flag_frequency.png` | `data/fixtures/final_report_metrics.json` |


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

FIGURES_DIR = ROOT / "figures"
FIXTURES_DIR = ROOT / "data" / "fixtures"
FIGURES_DIR.mkdir(exist_ok=True)

with open(FIXTURES_DIR / "final_report_metrics.json") as f:
    metrics = json.load(f)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
})

print(f"Repository root: {ROOT}")
print(f"Figures will be saved to: {FIGURES_DIR}")


---

## Figure 1: Training Dynamics

Training log history from the LoRA run. The vertical lines mark the EMA-minimum checkpoint and final checkpoint discussed in the report.


In [ ]:
with open(FIXTURES_DIR / "trainer_state.json") as f:
    trainer_state = json.load(f)

history = trainer_state.get("log_history", [])
loss_points = [(row["step"], row["loss"]) for row in history if "step" in row and "loss" in row]
eval_points = [(row["step"], row["eval_loss"]) for row in history if "step" in row and "eval_loss" in row]

steps = [p[0] for p in loss_points]
losses = [p[1] for p in loss_points]

alpha = 0.12
ema = []
for loss in losses:
    ema.append(loss if not ema else alpha * loss + (1 - alpha) * ema[-1])

fig, ax = plt.subplots(figsize=(11.5, 4.2))
ax.plot(steps, losses, color="#9ca3af", linewidth=1, alpha=0.45, label="training loss")
ax.plot(steps, ema, color="#2563eb", linewidth=2.4, label="training loss EMA")
if eval_points:
    ax.scatter([p[0] for p in eval_points], [p[1] for p in eval_points], color="#111827", s=28, label="eval loss")
ax.axvline(3000, color="#16a34a", linestyle="--", linewidth=1.8, label="checkpoint 3000")
ax.axvline(3690, color="#dc2626", linestyle="--", linewidth=1.8, label="checkpoint 3690")
ax.set_title("Training dynamics: earlier checkpoint beat final checkpoint")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.legend(ncol=2, fontsize=9)
fig.tight_layout()
out = FIGURES_DIR / "figure1_training_dynamics.png"
fig.savefig(out, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()


---

## Figure 2: Prompt Length Distribution

Reduced aggregate prompt-length bins show why document fragmentation mattered: a large share of raw prompts exceeded the 32K context target.


In [ ]:
dist = metrics["prompt_length_distribution"]
bins = dist["bins"]
labels = [b["label"] for b in bins]
counts = [b["count"] for b in bins]
total = sum(counts)
cumulative = []
running = 0
for count in counts:
    running += count
    cumulative.append(running / total)

fig, ax1 = plt.subplots(figsize=(11.5, 4.3))
ax1.bar(labels, counts, color="#2563eb", alpha=0.78)
ax1.set_ylabel("Cases")
ax1.set_xlabel("Prompt length bin (tokens)")
ax1.set_title("Prompt length distribution and context-window pressure")
ax2 = ax1.twinx()
ax2.plot(labels, cumulative, color="#111827", marker="o", linewidth=2, label="cumulative share")
ax2.axhline(dist["raw_fit_rate"], color="#dc2626", linestyle="--", linewidth=1.5, label="raw fit rate")
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("Cumulative share")
ax2.legend(loc="lower right")
fig.tight_layout()
out = FIGURES_DIR / "figure2_prompt_length_distribution.png"
fig.savefig(out, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()


---

## Figure 3: Checkpoint Comparison

Checkpoint 3000 and checkpoint 3690 are compared on reference metrics and downstream QA/grounding signals. Lower rejection and malformed-date rates are better.


In [ ]:
rows = metrics["checkpoint_comparison"]
labels = [r["system"] for r in rows]
series = [
    ("ROUGE-1", [r["rouge1"] for r in rows], "#2563eb"),
    ("QA pass", [r["qa_pass_rate"] for r in rows], "#16a34a"),
    ("Primary attribution", [r["primary_attribution"] for r in rows], "#0f766e"),
    ("QA reject", [r["qa_reject_rate"] for r in rows], "#dc2626"),
    ("Garbled dates", [r["garbled_date_rate"] for r in rows], "#f97316"),
]

x = range(len(labels))
width = 0.15
fig, ax = plt.subplots(figsize=(10.5, 4.6))
for i, (name, values, color) in enumerate(series):
    offsets = [v + (i - 2) * width for v in x]
    ax.bar(offsets, values, width=width, label=name, color=color, alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel("Rate / score")
ax.set_title("Checkpoint comparison: final checkpoint overfit despite similar reference metrics")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
out = FIGURES_DIR / "figure3_checkpoint_comparison.png"
fig.savefig(out, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()


---

## Figure 4: QA Triage Across Systems

Structural QA routes outputs into `PASS`, `REVIEW`, and `REJECT`. This is the operational layer that catches malformed dates, prompt leakage, missing legal elements, and length collapse.


In [ ]:
rows = metrics["qa_triage"]
systems = [r["system"] for r in rows]
pass_vals = [r["pass"] for r in rows]
review_vals = [r["review"] for r in rows]
reject_vals = [r["reject"] for r in rows]

def pct(vals):
    return [v / 50 * 100 for v in vals]

fig, ax = plt.subplots(figsize=(11.5, 4.6))
ax.bar(systems, pct(pass_vals), color="#16a34a", label="PASS")
ax.bar(systems, pct(review_vals), bottom=pct(pass_vals), color="#f59e0b", label="REVIEW")
bottom = [a + b for a, b in zip(pct(pass_vals), pct(review_vals))]
ax.bar(systems, pct(reject_vals), bottom=bottom, color="#dc2626", label="REJECT")
ax.set_ylabel("Share of 50-case sample (%)")
ax.set_title("QA triage status by generation system")
ax.legend(ncol=3)
ax.tick_params(axis="x", rotation=12)
fig.tight_layout()
out = FIGURES_DIR / "figure4_qa_triage_3systems.png"
fig.savefig(out, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()


---

## Figure 5: Source Attribution

Sentence-level attribution classifies generated sentences as strongly grounded in primary sources, weakly supported, or unsupported.


In [ ]:
rows = metrics["source_attribution"]
systems = [r["system"] for r in rows]
primary = [r["primary"] * 100 for r in rows]
weak = [r["weak_supported"] * 100 for r in rows]
unsupported = [r["unsupported"] * 100 for r in rows]

fig, ax = plt.subplots(figsize=(10.5, 4.5))
ax.bar(systems, primary, color="#16a34a", label="PRIMARY")
ax.bar(systems, weak, bottom=primary, color="#f59e0b", label="SUPPORTED")
bottom = [a + b for a, b in zip(primary, weak)]
ax.bar(systems, unsupported, bottom=bottom, color="#dc2626", label="UNSUPPORTED")
ax.set_ylabel("Sentence share (%)")
ax.set_title("Source attribution of generated sentences")
ax.legend(ncol=3)
fig.tight_layout()
out = FIGURES_DIR / "figure5_source_attribution.png"
fig.savefig(out, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()


---

## Figure 6: Cost vs. Quality

Claude is the measured quality ceiling, while self-hosted and metadata-only routes are effectively zero marginal API cost after setup.


In [ ]:
rows = metrics["cost_quality"]
fig, ax = plt.subplots(figsize=(10.5, 5.0))
for r in rows:
    x = r["cost_per_case_usd"]
    y = r["rouge1"]
    size = 120 + 500 * r["qa_pass_rate"]
    color = "#2563eb" if x == 0 else "#7c3aed"
    ax.scatter(x, y, s=size, alpha=0.78, color=color, edgecolor="#111827")
    ax.annotate(r["system"], (x, y), textcoords="offset points", xytext=(8, 5), fontsize=9)
ax.set_xlabel("Estimated API cost per case (USD)")
ax.set_ylabel("Mean ROUGE-1")
ax.set_title("Cost vs. reference quality on the 50-case sample")
ax.set_xlim(-0.03, 0.5)
ax.set_ylim(0.32, 0.49)
fig.tight_layout()
out = FIGURES_DIR / "figure6_cost_quality.png"
fig.savefig(out, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()


---

## Figure 7: QA Flag Frequency

Top structural QA flags for the overfit checkpoint. The high `GARBLED_DATE` count is the clearest fingerprint of checkpoint 3690's degradation.


In [ ]:
rows = metrics["flag_frequency"]
colors = {"critical": "#dc2626", "warning": "#f59e0b", "info": "#6b7280"}
labels = [r["code"] for r in rows]
counts = [r["count"] for r in rows]
bar_colors = [colors[r["severity"]] for r in rows]

fig, ax = plt.subplots(figsize=(11.5, 5.0))
ax.barh(labels[::-1], counts[::-1], color=bar_colors[::-1])
ax.set_xlabel("Flag count across 50 summaries")
ax.set_title("Top QA failure modes for checkpoint 3690")
fig.tight_layout()
out = FIGURES_DIR / "figure7_flag_frequency.png"
fig.savefig(out, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()


---

## Export Check

This cell confirms that every final report figure exists.


In [ ]:
expected = [
    "figure1_training_dynamics.png",
    "figure2_prompt_length_distribution.png",
    "figure3_checkpoint_comparison.png",
    "figure4_qa_triage_3systems.png",
    "figure5_source_attribution.png",
    "figure6_cost_quality.png",
    "figure7_flag_frequency.png",
]
for name in expected:
    path = FIGURES_DIR / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")
